## Einsum

In [56]:
import numpy as np

In [57]:
# Exercise 6.1
np.random.seed(42)

a = np.random.randn(3)
b = np.random.randn(3)
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C = np.random.randn(3, 4, 5)
D = np.random.randn(3, 4, 5)

In [58]:
assert np.isclose(np.dot(a, b), np.einsum("i,i->", a, b))
assert np.allclose(a[:, np.newaxis] * b, np.einsum("i,j->ij", a, b))
assert np.allclose(A @ B, np.einsum("ij,jk->ik", A, B))
assert np.allclose(A.T, np.einsum("ij->ji", A))
assert np.allclose(A.sum(axis=0), np.einsum("ij->j", A))
assert np.allclose(A.sum(axis=1), np.einsum("ij->i", A))
assert np.allclose((A * A).sum(), np.einsum("ij,ij->", A, A))
assert np.allclose(np.diag(A @ A.T), np.einsum("ij,ij->i", A, A))

In [59]:
# Exercise 6.2
np.random.seed(1)

batch_A = np.random.randn(8, 3, 4)
batch_B = np.random.randn(8, 4, 5)

w = np.random.randn(4)

In [60]:
# a)
batch_matmul = np.einsum("bij,bjk->bik", batch_A, batch_B)
assert np.allclose(batch_matmul, np.stack([batch_A[i] @ batch_B[i] for i in range(8)]))

# b)
batch_dot = np.einsum("bij,j->bi", batch_A, w)
print(batch_dot.shape)

# c)
batch_gram = np.einsum("bij,bik->bjk", batch_A, batch_A)
print(batch_gram.shape)

(8, 3)
(8, 4, 4)


In [61]:
# Exercise 6.3
np.random.seed(42)
batch_size = 2
seq_len = 5
d_k = 8

Q = np.random.randn(batch_size, seq_len, d_k)
K = np.random.randn(batch_size, seq_len, d_k)
V = np.random.randn(batch_size, seq_len, d_k)

In [62]:
def softmax(logits):
    shifted = logits - logits.max(axis=-1, keepdims=True)
    shifted_exp = np.exp(shifted)
    return shifted_exp / shifted_exp.sum(axis=-1, keepdims=True)

scores = np.einsum("bij, bkj->bik", Q, K)
scores = scores / np.sqrt(d_k)
attention_weights = softmax(scores)
weight_sum = np.einsum("bij,bjk->bik", attention_weights, V)
assert np.allclose(attention_weights.sum(axis=-1), 1)